# Forecast California Housing Prices

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Dataset
from sklearn.datasets import fetch_california_housing

In [3]:
np.random.seed(50)
torch.manual_seed(50)

In [4]:
# Load the dataset
data = fetch_california_housing()
features = pd.DataFrame(data.data, columns=data.feature_names)
targets = pd.Series(data.target, name='Price').values.astype(float).reshape(-1, 1)

In [5]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

features_normalized = scaler_X.fit_transform(features)
targets_normalized = scaler_y.fit_transform(targets)

**Questions**

* What is a `Bunch`?
* What is the default type for `data`?
* What is the default type for `target`?

**Answers**

1.  A Bunch is a dictionary-like object used in sklearn that allows access to data using both keys and attributes.
2. The default type for data is a NumPy array.
3. The default type for target is also a NumPy array.


In [6]:
def create_sequences(data, targets, seq_length):
   X, y = [], []
   for i in range(len(data) - seq_length):
       X.append(data[i:i+seq_length])
       y.append(targets[i+seq_length])
   return np.array(X), np.array(y)


seq_length = 20
X, y = create_sequences(features_normalized, targets_normalized, seq_length)

In [7]:
# Split data
train_size = int(len(X) * 0.67)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Data cleaning and preprocessing

# LSTM model

In [8]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.lstm(x)
        output = self.fc(output[:, -1, :])
        return output


# Hyperparameters
input_size = X_train.shape[2]
hidden_size = 50
output_size = 1
num_layers = 2
learning_rate = 0.001
num_epochs = 100
l2_penalty = 0.01

# Model, loss function, and optimizer
lstm_model = LSTMModel(input_size, hidden_size, output_size, num_layers)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=learning_rate, weight_decay=l2_penalty)

In [9]:
# Train the LSTM model
for epoch in range(num_epochs):
    optimizer.zero_grad()
    output = lstm_model(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [1/100], Loss: 0.1573
Epoch [2/100], Loss: 0.1454
Epoch [3/100], Loss: 0.1342
Epoch [4/100], Loss: 0.1236
Epoch [5/100], Loss: 0.1138
Epoch [6/100], Loss: 0.1045
Epoch [7/100], Loss: 0.0959
Epoch [8/100], Loss: 0.0880
Epoch [9/100], Loss: 0.0807
Epoch [10/100], Loss: 0.0741
Epoch [11/100], Loss: 0.0684
Epoch [12/100], Loss: 0.0635
Epoch [13/100], Loss: 0.0595
Epoch [14/100], Loss: 0.0565
Epoch [15/100], Loss: 0.0545
Epoch [16/100], Loss: 0.0533
Epoch [17/100], Loss: 0.0528
Epoch [18/100], Loss: 0.0529
Epoch [19/100], Loss: 0.0533
Epoch [20/100], Loss: 0.0538
Epoch [21/100], Loss: 0.0542
Epoch [22/100], Loss: 0.0545
Epoch [23/100], Loss: 0.0545
Epoch [24/100], Loss: 0.0543
Epoch [25/100], Loss: 0.0540
Epoch [26/100], Loss: 0.0536
Epoch [27/100], Loss: 0.0532
Epoch [28/100], Loss: 0.0528
Epoch [29/100], Loss: 0.0526
Epoch [30/100], Loss: 0.0525
Epoch [31/100], Loss: 0.0525
Epoch [32/100], Loss: 0.0526
Epoch [33/100], Loss: 0.0527
Epoch [34/100], Loss: 0.0529
Epoch [35/100], Loss: 0

In [10]:
# Make predictions
lstm_model.eval()

with torch.no_grad():
    predictions = lstm_model(X_test_tensor).numpy()

# Inverse scaling
predictions_rescaled = scaler_y.inverse_transform(predictions)
y_test_rescaled = scaler_y.inverse_transform(y_test)

# Calculate MSE
mse = mean_squared_error(y_test_rescaled, predictions_rescaled)

print("LSTM Mean Squared Error:", mse)

LSTM Mean Squared Error: 1.4598739866105064


# GRU model

In [11]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.gru(x)
        output = self.fc(output[:, -1, :])
        return output


# Hyperparameters (same as LSTM)
gru_model = GRUModel(input_size, hidden_size, output_size, num_layers)
criterion = nn.MSELoss()

# ❗ NO regularisation here
optimizer = optim.Adam(gru_model.parameters(), lr=learning_rate)

In [12]:
# Train the GRU model
for epoch in range(num_epochs):
    optimizer.zero_grad()
    output = gru_model(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [1/100], Loss: 0.2840
Epoch [2/100], Loss: 0.2362
Epoch [3/100], Loss: 0.1936
Epoch [4/100], Loss: 0.1561
Epoch [5/100], Loss: 0.1238
Epoch [6/100], Loss: 0.0968
Epoch [7/100], Loss: 0.0755
Epoch [8/100], Loss: 0.0606
Epoch [9/100], Loss: 0.0525
Epoch [10/100], Loss: 0.0513
Epoch [11/100], Loss: 0.0560
Epoch [12/100], Loss: 0.0636
Epoch [13/100], Loss: 0.0701
Epoch [14/100], Loss: 0.0730
Epoch [15/100], Loss: 0.0721
Epoch [16/100], Loss: 0.0685
Epoch [17/100], Loss: 0.0637
Epoch [18/100], Loss: 0.0589
Epoch [19/100], Loss: 0.0549
Epoch [20/100], Loss: 0.0521
Epoch [21/100], Loss: 0.0506
Epoch [22/100], Loss: 0.0501
Epoch [23/100], Loss: 0.0503
Epoch [24/100], Loss: 0.0510
Epoch [25/100], Loss: 0.0518
Epoch [26/100], Loss: 0.0525
Epoch [27/100], Loss: 0.0530
Epoch [28/100], Loss: 0.0532
Epoch [29/100], Loss: 0.0531
Epoch [30/100], Loss: 0.0527
Epoch [31/100], Loss: 0.0520
Epoch [32/100], Loss: 0.0511
Epoch [33/100], Loss: 0.0502
Epoch [34/100], Loss: 0.0494
Epoch [35/100], Loss: 0

In [21]:
# Make predictions with GRU
gru_model.eval()

with torch.no_grad():
    gru_predictions = gru_model(X_test_tensor).numpy()

# Inverse scaling
gru_predictions_rescaled = scaler_y.inverse_transform(gru_predictions)
y_test_rescaled = scaler_y.inverse_transform(y_test)

# Calculate MSE
gru_mse = mean_squared_error(y_test_rescaled, gru_predictions_rescaled)

print("GRU Mean Squared Error:", gru_mse)
print("LSTM Without Regularisation MSE:", lstm_no_reg_mse)

GRU Mean Squared Error: 0.9370349887731871
LSTM Without Regularisation MSE: 0.768087472349114


## LSTM Without Regularisation

In [19]:
# LSTM without regularisation
lstm_no_reg = LSTMModel(input_size, hidden_size, output_size, num_layers)

criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_no_reg.parameters(), lr=learning_rate)

# Train the model
for epoch in range(num_epochs):
    optimizer.zero_grad()
    output = lstm_no_reg(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [1/100], Loss: 0.1759
Epoch [2/100], Loss: 0.1607
Epoch [3/100], Loss: 0.1470
Epoch [4/100], Loss: 0.1347
Epoch [5/100], Loss: 0.1235
Epoch [6/100], Loss: 0.1132
Epoch [7/100], Loss: 0.1036
Epoch [8/100], Loss: 0.0947
Epoch [9/100], Loss: 0.0865
Epoch [10/100], Loss: 0.0788
Epoch [11/100], Loss: 0.0719
Epoch [12/100], Loss: 0.0657
Epoch [13/100], Loss: 0.0605
Epoch [14/100], Loss: 0.0566
Epoch [15/100], Loss: 0.0542
Epoch [16/100], Loss: 0.0539
Epoch [17/100], Loss: 0.0555
Epoch [18/100], Loss: 0.0582
Epoch [19/100], Loss: 0.0602
Epoch [20/100], Loss: 0.0607
Epoch [21/100], Loss: 0.0598
Epoch [22/100], Loss: 0.0582
Epoch [23/100], Loss: 0.0565
Epoch [24/100], Loss: 0.0550
Epoch [25/100], Loss: 0.0539
Epoch [26/100], Loss: 0.0532
Epoch [27/100], Loss: 0.0530
Epoch [28/100], Loss: 0.0529
Epoch [29/100], Loss: 0.0530
Epoch [30/100], Loss: 0.0532
Epoch [31/100], Loss: 0.0534
Epoch [32/100], Loss: 0.0536
Epoch [33/100], Loss: 0.0536
Epoch [34/100], Loss: 0.0536
Epoch [35/100], Loss: 0

In [20]:
# Predictions without regularisation
lstm_no_reg.eval()

with torch.no_grad():
    lstm_no_reg_predictions = lstm_no_reg(X_test_tensor).numpy()

# Inverse scaling
lstm_no_reg_predictions_rescaled = scaler_y.inverse_transform(lstm_no_reg_predictions)
y_test_rescaled = scaler_y.inverse_transform(y_test)

# MSE
lstm_no_reg_mse = mean_squared_error(y_test_rescaled, lstm_no_reg_predictions_rescaled)

print("LSTM Without Regularisation MSE:", lstm_no_reg_mse)

LSTM Without Regularisation MSE: 0.768087472349114


## Comparison without Regularisation

### a. Has performance gotten worse in both models?
No. In this case, performance did not get worse in both models. The LSTM model without regularisation performed better than the GRU model without regularisation because it achieved a lower mean squared error (LSTM ≈ 0.768 vs GRU ≈ 0.937) on the test data. This shows that removing regularisation did not negatively affect both models equally.

### b. What is the importance of regularisation for optimising the efficiency of models?
Regularisation is important because it helps prevent overfitting. It discourages the model from relying too much on specific patterns in the training data, which improves generalisation to unseen data. In sequential models such as LSTM and GRU, regularisation can also improve training stability and help the model learn more meaningful patterns instead of memorising the training set.

## Hyperparameter Tuning with Regularisation

In [14]:
# LSTM Config 1 (stronger regularisation)
lstm_model_1 = LSTMModel(input_size, 100, output_size, 2)

criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_model_1.parameters(), lr=0.001, weight_decay=0.01)

# Train
for epoch in range(50):
    optimizer.zero_grad()
    output = lstm_model_1(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

# Evaluate
lstm_model_1.eval()
with torch.no_grad():
    preds = lstm_model_1(X_test_tensor).numpy()

preds_rescaled = scaler_y.inverse_transform(preds)
mse_lstm_1 = mean_squared_error(y_test_rescaled, preds_rescaled)

print("LSTM Config 1 MSE:", mse_lstm_1)

LSTM Config 1 MSE: 1.471802432052384


In [15]:
# LSTM Config 2 (weaker regularisation + more layers)
lstm_model_2 = LSTMModel(input_size, 50, output_size, 3)

optimizer = optim.Adam(lstm_model_2.parameters(), lr=0.0005, weight_decay=0.001)

for epoch in range(50):
    optimizer.zero_grad()
    output = lstm_model_2(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

lstm_model_2.eval()
with torch.no_grad():
    preds = lstm_model_2(X_test_tensor).numpy()

preds_rescaled = scaler_y.inverse_transform(preds)
mse_lstm_2 = mean_squared_error(y_test_rescaled, preds_rescaled)

print("LSTM Config 2 MSE:", mse_lstm_2)

LSTM Config 2 MSE: 1.50403196242776


In [16]:
# GRU Config 1 (with regularisation)
gru_model_1 = GRUModel(input_size, 100, output_size, 2)

optimizer = optim.Adam(gru_model_1.parameters(), lr=0.001, weight_decay=0.01)

for epoch in range(50):
    optimizer.zero_grad()
    output = gru_model_1(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

gru_model_1.eval()
with torch.no_grad():
    preds = gru_model_1(X_test_tensor).numpy()

preds_rescaled = scaler_y.inverse_transform(preds)
mse_gru_1 = mean_squared_error(y_test_rescaled, preds_rescaled)

print("GRU Config 1 MSE:", mse_gru_1)

GRU Config 1 MSE: 1.3927610137544155


In [17]:
# GRU Config 2 (different learning rate)
gru_model_2 = GRUModel(input_size, 50, output_size, 3)

optimizer = optim.Adam(gru_model_2.parameters(), lr=0.0005, weight_decay=0.001)

for epoch in range(50):
    optimizer.zero_grad()
    output = gru_model_2(X_train_tensor)
    loss = criterion(output, y_train_tensor)
    loss.backward()
    optimizer.step()

gru_model_2.eval()
with torch.no_grad():
    preds = gru_model_2(X_test_tensor).numpy()

preds_rescaled = scaler_y.inverse_transform(preds)
mse_gru_2 = mean_squared_error(y_test_rescaled, preds_rescaled)

print("GRU Config 2 MSE:", mse_gru_2)

GRU Config 2 MSE: 1.4279210925248678


## Final Comparison of Hyperparameters

Different hyperparameter configurations had a noticeable impact on model performance.

For the LSTM model, increasing the hidden size and using stronger regularisation improved stability, while changing the learning rate and number of layers affected convergence speed and performance.

For the GRU model, both configurations performed well, but adjusting the learning rate and model complexity influenced the final mean squared error.

Overall, the GRU model continued to outperform the LSTM model across different configurations, indicating that it is more suitable for this dataset. Regularisation helped stabilise training and improve generalisation across both models.

The results suggest that model selection and hyperparameter tuning play a critical role in achieving optimal performance in sequential learning tasks.

**Note:** The California housing dataset was sourced by scikit-learn from the StatLib repository: https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

**Reference**  
Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. Statistics & Probability Letters, 33(3), 291-297.